# Autonomous Executive Assistant Sandbox

Notebook for deterministic evaluation, OpenAI policy runs, and rollout export. Keep durable logic in `src/executive_assistant/` and use this notebook to inspect traces and bootstrap training data.

## Workflow

1. Choose a scenario and policy provider.
2. Run the baseline suite to verify the environment is stable.
3. Set `OPENAI_API_KEY` and switch to the OpenAI policy.
4. Export episode traces to JSONL for imitation or reward-model experiments.
5. Promote any stable logic back into `src/` and cover it with tests.

In [ ]:
import json
import os
from pathlib import Path

from src.executive_assistant.agent import BaselineAgent, OpenAIResponsesPolicy
from src.executive_assistant.runner import EpisodeRunner, export_traces_jsonl, run_policy_suite


In [ ]:
TASK_NAME = "hard_rag_reply"
POLICY_PROVIDER = "baseline"  # change to "openai" after setting OPENAI_API_KEY
MODEL_NAME = "gpt-4.1-mini"
MAX_STEPS = 12
TRACE_DIR = Path("artifacts/traces")
TRACE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def build_policy(provider: str, model_name: str):
    if provider == "baseline":
        return BaselineAgent()
    if provider == "openai":
        api_key = os.environ.get("OPENAI_API_KEY")
        if not api_key:
            raise RuntimeError("Set OPENAI_API_KEY before using the OpenAI policy.")
        return OpenAIResponsesPolicy(api_key=api_key, model_name=model_name)
    raise ValueError(f"Unsupported provider: {provider}")


## Baseline validation

Run this first. If the baseline does not still solve the seeded tasks, do not trust later OpenAI results.

In [ ]:
baseline_traces = run_policy_suite(
    policy=BaselineAgent(),
    task_names=[
        "easy_deadline_extraction",
        "medium_triage_and_negotiation",
        "hard_rag_reply",
    ],
    max_steps=MAX_STEPS,
)

{name: {"completed": trace.completed, "score": trace.final_score, "steps": len(trace.steps)} for name, trace in baseline_traces.items()}


## Single-episode trace

Use this for prompt iteration and for inspecting the exact reward progression step by step.

In [ ]:
policy = build_policy(POLICY_PROVIDER, MODEL_NAME)
runner = EpisodeRunner(policy=policy, max_steps=MAX_STEPS)
trace = runner.run(TASK_NAME)

print(json.dumps(trace.to_dict(), indent=2))


In [ ]:
trace.steps[-1].snapshot


## Export rollouts

These JSONL traces are the starting point for imitation-learning style training, reward analysis, or regression playback.

In [ ]:
suite_traces = run_policy_suite(
    policy=build_policy(POLICY_PROVIDER, MODEL_NAME),
    task_names=[TASK_NAME],
    max_steps=MAX_STEPS,
)

output_path = export_traces_jsonl(
    list(suite_traces.values()),
    TRACE_DIR / f"{POLICY_PROVIDER}_{TASK_NAME}_traces.jsonl",
)

print(output_path)


## OpenAI setup note

Before switching `POLICY_PROVIDER` to `"openai"`, set the key in the notebook kernel environment, for example:

```python
import os
os.environ["OPENAI_API_KEY"] = "..."
```

Then rerun the config and execution cells. The same runner and export path will work without further code changes.